### 4. ✅ Class-Based Decorator with State — (Advanced OOP)
Shows you know decorators aren't just functions.

**Why it impresses**: Circuit Breaker is a famous microservices resilience pattern, class-based decorator shows deep OOP + decorator mastery.

In [ ]:
import functools
import time

class CircuitBreaker:
    """
    Prevents cascading failures by stopping calls to a failing service.
    Real use case: microservices, external API integrations.
    States: CLOSED (normal) → OPEN (blocking) → HALF-OPEN (testing)
    """
    def __init__(self, failure_threshold=3, recovery_timeout=10):
        self.failure_threshold = failure_threshold
        self.recovery_timeout  = recovery_timeout
        self.failure_count     = 0
        self.last_failure_time = None
        self.state             = "CLOSED"  # CLOSED, OPEN, HALF-OPEN

    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if self.state == "OPEN":
                elapsed = time.monotonic() - self.last_failure_time
                if elapsed >= self.recovery_timeout:
                    self.state = "HALF-OPEN"
                    print("Circuit HALF-OPEN: testing service...")
                else:
                    raise Exception(
                        f"Circuit OPEN. Retry after "
                        f"{self.recovery_timeout - elapsed:.1f}s"
                    )
            try:
                result = func(*args, **kwargs)
                self._on_success()
                return result
            except Exception as e:
                self._on_failure()
                raise
        return wrapper

    def _on_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def _on_failure(self):
        self.failure_count += 1
        self.last_failure_time = time.monotonic()
        if self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            print(f"Circuit OPEN after {self.failure_count} failures!")


# --- Usage ---
circuit_breaker = CircuitBreaker(failure_threshold=3, recovery_timeout=10)

@circuit_breaker
def call_payment_service(amount: float):
    raise ConnectionError("Payment service down!")  # simulate failure

for i in range(6):
    try:
        call_payment_service(99.99)
    except Exception as e:
        print(f"Call {i+1}: {e}")